In [ ]:
# Hack: empty cell to allow Omnipy to detect Jupyter embedded within PyCharm (see https://github.com/fairtracks/omnipy/issues/244). Run this cell manually, then wait briefly and run the next cells.

In [ ]:
from omnipy import Dataset, Model
import omnipy as om
om.setup_jupyter_ui()

In [ ]:
output = om.JsonDataset()

In [ ]:
om.runtime.config.engine.choice = 'prefect'
om.runtime.config.root_log.log_to_stdout = False

In [ ]:
from omnipy.components.remote.tasks import get_auto_from_api_endpoint
task = get_auto_from_api_endpoint.refine(
    output_dataset_param='output_dataset').run(
        om.HttpUrlDataset(a='https://seqcolapi.databio.org/list/collection'),
        output_dataset=output,
    )

In [ ]:
await task

In [ ]:
output[0]

In [ ]:
urls = om.HttpUrlDataset({_:f'https://seqcolapi.databio.org/collection/{_}' for _ in output[0]['results']})

In [ ]:
import asyncio
from typing import AsyncGenerator, cast
from omnipy.components.remote.tasks import _ensure_retry_session, _check_response_status, _call_get
from aiohttp import ClientSession
from aiohttp_retry import RetryClient
from omnipy.data.helpers import FailedData

@om.TaskTemplate()
async def get_json_from_api_endpoint(
    urls: om.HttpUrlDataset,
    retry_client: RetryClient | None = None,
) -> AsyncGenerator[tuple[str, om.JsonModel | FailedData]]:
    
    async def fetch_one(key, url):
        try:
            async for retry_session in _ensure_retry_session(retry_client):
                async for response in _call_get(url, cast(ClientSession, retry_session)):
                    _check_response_status(response)
                    return key, om.JsonModel(await response.json(content_type=None))
        except Exception as exp:
            return key, FailedData(job_name=key, job_unique_name=key, exception=exp)

    # 1. CRITICAL: Explicitly wrap them in asyncio.create_task so we hold strong references
    tasks = [asyncio.create_task(fetch_one(key, url)) for key, url in urls.items()]
    
    try:
        # Yield the raw chunks as they finish
        for finished_task in asyncio.as_completed(tasks):
            key, json_model = await finished_task
            yield key, json_model
            
    except asyncio.CancelledError:
        # 2. FIX: The consumer cancelled this generator mid-execution. 
        # We MUST clean up the remaining background tasks.
        for t in tasks:
            if not t.done():
                t.cancel()
        
        # 3. FIX: Await all remaining tasks with return_exceptions=True 
        # This officially "retrieves" the exceptions/cancellations so Python stops complaining.
        await asyncio.gather(*tasks, return_exceptions=True)
        
        # 4. Propagate the cancellation up to let the framework know we exited cleanly
        raise


In [ ]:
import asyncio
from IPython.display import display
from omnipy.shared.exceptions import FailedDataError, PendingDataError
from omnipy.data.helpers import PendingData, FailedData
from aiohttp_retry import RetryClient
from omnipy.components.remote.tasks import get_retry_client
from omnipy.components.remote.helpers import RateLimitingClientSession
from copy import copy
from random import randint

class LiveJsonDataset:
    def __init__(self, urls, retry_client=None):
        self.urls = urls
        self.retry_client = retry_client
        if self.retry_client is None:
            host_config = om.runtime.config.data.http.for_host['seqcolapi.databio.org']
            
            self.client_session = RateLimitingClientSession(
                    host_config.requests_per_time_period,
                    host_config.time_period_in_secs
            )
            self.retry_client = get_retry_client(
                client_session=self.client_session,
                retry_http_statuses=host_config.retry_http_statuses,
                retry_attempts=host_config.retry_attempts,
                retry_backoff_strategy=host_config.retry_backoff_strategy
            )
        
        # 1. Initialize the dataset with pending states immediately
        self.dataset = om.JsonDataset()
        
        for title, data_file in urls.items():
            self.dataset[title] = PendingData(job_name=title, job_unique_name=title)

        # 2. Kick off the background fetch instantly
        # (Runs concurrently without blocking the Jupyter cell execution)
        self._task = asyncio.create_task(self._fetch_loop())

    async def _fetch_loop(self):
        # Your original concurrency logic feeding into the generator
        i=0
        async for key, real_data in get_json_from_api_endpoint.run(self.urls, self.retry_client):
            self.dataset[key] = real_data
            i+=1
            # 3. Force Jupyter to re-render this specific object visual
            self._update_display(i)

    def _update_display(self, i):
        self.dataset.update_reactive_views()

    def _ipython_display_(self):
        self.dataset._ipython_display_()


In [ ]:
om.runtime.config.data.http.for_host['seqcolapi.databio.org'].requests_per_time_period = 10
om.runtime.config.data.http.for_host['seqcolapi.databio.org'].time_period_in_secs = 2
om.runtime.config.data.http.for_host['seqcolapi.databio.org'].retry_attempts = 10
om.runtime.config.data.http.for_host['seqcolapi.databio.org'].retry_backoff_strategy = 'fibonacci'

In [ ]:
live_data = LiveJsonDataset(urls)
live_data

In [ ]:
await live_data._task
live_data.dataset[43].full(printer='compact-json', style=om.DarkHighContrastTintedThemingBase16ColorStyles.UWUNICORN_T16)